# Pull Tweets by Keyword
Search X (Twitter) for a keyword and load results into a pandas DataFrame.

In [20]:
# Install dependencies if needed
# %pip install tweepy pandas google-cloud-secret-manager

In [21]:
import tweepy
import pandas as pd
from google.cloud import secretmanager

# ── Configuration ────────────────────────────────────────────────────────────
GCP_PROJECT_ID = "datalogichub"   # Your GCP project ID

KEYWORD  = "python"   # Search keyword / hashtag
N_TWEETS = 50          # Number of tweets to pull (min 10, max 100 per request)
# ─────────────────────────────────────────────────────────────────────────────


def get_secret(secret_id: str, project_id: str = GCP_PROJECT_ID) -> str:
    """Fetch the latest version of a secret from GCP Secret Manager."""
    sm_client = secretmanager.SecretManagerServiceClient()
    name = f"projects/{project_id}/secrets/{secret_id}/versions/latest"
    response = sm_client.access_secret_version(request={"name": name})
    return response.payload.data.decode("UTF-8")


# Load Bearer Token from Secret Manager (app-only auth — required for search)
bearer_token = get_secret("MONGOL_BEARER_TOKEN")

tweepy_v2 = tweepy.Client(bearer_token=bearer_token, wait_on_rate_limit=True)

print("Client initialised.")

Client initialised.


In [28]:
def pull_tweets(keyword: str, n: int, top_by_favorites: bool = False) -> pd.DataFrame:
    """
    Search X for `keyword` and return `n` tweets as a DataFrame.

    top_by_favorites=True: fetches up to 100 recent tweets and returns
    the top `n` sorted by favorites (likes). The X API does not support
    native sort-by-likes, so this is a client-side sort over a larger batch.
    """
    rows = []
    fetched = 0

    # When sorting by favorites, fetch the max batch (100) to rank from
    fetch_limit = 100 if top_by_favorites else min(max(n, 10), 100)

    tweet_fields = ["created_at", "public_metrics", "author_id"]
    user_fields  = ["username"]
    expansions   = ["author_id"]

    paginator = tweepy.Paginator(
        tweepy_v2.search_recent_tweets,
        query=f"{keyword} -is:retweet",
        tweet_fields=tweet_fields,
        user_fields=user_fields,
        expansions=expansions,
        max_results=fetch_limit,
        limit=1,  # one page — avoids unnecessary extra API calls
    )

    for response in paginator:
        if not response.data:
            break

        users = {u.id: u.username for u in (response.includes.get("users") or [])}

        for tweet in response.data:
            metrics = tweet.public_metrics or {}
            rows.append({
                "username":    users.get(tweet.author_id, "unknown"),
                "post_text":   tweet.text,
                "posted_date": tweet.created_at,
                "retweets":    metrics.get("retweet_count", 0),
                "favorites":   metrics.get("like_count", 0),
            })
            fetched += 1
            if fetched >= fetch_limit:
                break

    df = pd.DataFrame(rows, columns=["username", "post_text", "posted_date", "retweets", "favorites"])
    df["posted_date"] = pd.to_datetime(df["posted_date"])

    return df

In [33]:
KEYWORD  = "Оюун-Эрдэнэ"   # Search keyword / hashtag
N_TWEETS = 500          # Number of tweets to pull (min 10, max 100 per request)

In [34]:
df = pull_tweets(KEYWORD, N_TWEETS)
print(f"Fetched {len(df)} tweets for keyword: '{KEYWORD}'")

Fetched 100 tweets for keyword: 'Оюун-Эрдэнэ'


In [35]:
df = df.sort_values("favorites", ascending=False).reset_index(drop=True)
df

,username,post_text,posted_date,retweets,favorites
0,Ariunq,Оюун-Эрдэнэ Энхбаярын талаар сайхан хэлж🤭👏🏻👏🏻Э...,2026-03-12 06:21:44+00:00,26,43
1,Baagij13310361,Сэтгүүлч: Д.Амарбаясгалан нарыг МАН-аас хөөхөд...,2026-03-12 07:00:10+00:00,14,38
2,miigaasbntv,Лувсаннамсрайн Оюун-Эрдэнэ шөнөжин ярьж Баттөм...,2026-03-12 08:53:52+00:00,16,34
3,iSee_MN,"Ерөнхий сайд асан Л.Оюун-Эрдэнэ ""хүүгийнхээ ор...",2026-03-12 09:21:46+00:00,15,32
4,newspressmn,Нийгмийн сэтгэл зүйч Г.Уянгаа: Ерөнхий сайд ас...,2026-03-12 13:27:15+00:00,23,31
...,...,...,...,...,...
95,Baagij13310361,Л.Оюун-Эрдэнэ: Өнөрөөг хонгилдсон нь үнэн. Үнд...,2026-03-12 05:38:45+00:00,0,0
96,Elch_mn,Л.Оюун-Эрдэнэ: Өнөрөөг хонгилдсон нь үнэн. Үнд...,2026-03-12 05:32:41+00:00,1,0
97,newswirmongolia,Бодь Интернэшнл: Ерөнхий сайд асан Л. Оюун-Эрд...,2026-03-12 05:23:22+00:00,0,0
98,Bolortat,Л.Оюун-Эрдэнэ: У.Хүрэлсүх даргаа галаар битгий...,2026-03-12 05:18:56+00:00,0,0


In [36]:
df.to_csv("dankh.csv", index = False)